## 1. Raw Data
Explain that we are loading the primary dataset.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load data
data_path = 'playground-series-s6e5' if os.path.exists('playground-series-s6e5') else '.'
train = pd.read_csv(f'{data_path}/train.csv')
test = pd.read_csv(f'{data_path}/test.csv')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')

## 2. Data Validation
Explain we are checking for missing values and data type consistency.

In [ ]:
print('Missing values in Train:')
print(train.isnull().sum())
print('\nData Types:')
print(train.dtypes)

## 3. Exploratory Data Analysis (EDA)
Visualizing the target distribution and primary features to establish a baseline understanding.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# PitNextLap distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='PitNextLap', data=train)
plt.title('Distribution of PitNextLap')
plt.show()

# TyreLife vs PitNextLap
plt.figure(figsize=(10, 6))
sns.kdeplot(data=train, x='TyreLife', hue='PitNextLap', fill=True)
plt.title('Distribution of TyreLife for Pit vs No-Pit')
plt.show()

## 4. Data Preprocessing
Standardizing column names and encoding categorical variables for model compatibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Clean column names (strip special chars)
train.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in train.columns]
test.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in test.columns]

cat_cols = ['Driver', 'Compound', 'Race']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[cat_cols] = encoder.fit_transform(train[cat_cols].astype(str))
test[cat_cols] = encoder.transform(test[cat_cols].astype(str))
print('Categorical columns encoded.')

## 5. Model Training
Selecting LightGBM as our baseline due to its speed and native GPU support.

In [ ]:
import lightgbm as lgb

# Define features and target
X = train.drop(['id', 'PitNextLap'], axis=1)
y = train['PitNextLap']

# Baseline LGBM params
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'device': 'gpu'
}

# Try GPU if possible, with a CPU fallback
try:
    # Small test to see if GPU works
    import numpy as np
    temp_X = np.random.rand(10, X.shape[1])
    temp_y = np.random.randint(0, 2, 10)
    temp_data = lgb.Dataset(temp_X, label=temp_y)
    lgb.train(params, temp_data, num_boost_round=1)
    print('Using GPU for LightGBM')
except Exception as e:
    print(f'Falling back to CPU: {e}')
    params['device'] = 'cpu'

model = lgb.LGBMClassifier(**params)

## 6. Model Evaluation
Evaluating the initial model on a simple hold-out set using ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

eval_model = lgb.LGBMClassifier(**params)
eval_model.fit(X_train, y_train)

val_preds = eval_model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)
print(f'ROC-AUC on hold-out set: {auc_score}')

## 7. Model Validation
Ensuring model stability using 5-Fold Stratified Cross-Validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
    
    m = lgb.LGBMClassifier(**params)
    m.fit(X_t, y_t)
    p = m.predict_proba(X_v)[:, 1]
    auc = roc_auc_score(y_v, p)
    aucs.append(auc)
    print(f'Fold {fold+1} AUC: {auc}')

print(f'Mean AUC: {np.mean(aucs):.5f} +/- {np.std(aucs):.5f}')

## 8. Model Deployment & Feedback
Outputting artifacts and the final submission file.

In [ ]:
import pickle

# Final model on full data
final_model = lgb.LGBMClassifier(**params)
final_model.fit(X, y)

# Save the model
with open('baseline_lgbm.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# Generate submission
test_X = test.drop(['id'], axis=1)
test_preds = final_model.predict_proba(test_X)[:, 1]
submission = pd.DataFrame({'id': test['id'], 'PitNextLap': test_preds})
submission.to_csv('submission.csv', index=False)
print('Saved baseline_lgbm.pkl and submission.csv')